In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style"
)

In [3]:
DATA_DIR = Path.cwd() / "../data/raw"
DATA_DIR = DATA_DIR.resolve()

DATA_DIR

PosixPath('/Users/aserra/projects/timeseries/energy-load-forecast/data/raw')

In [4]:
files = sorted(DATA_DIR.glob("*.xlsx"))

for file in files:
    print(file.name)

dfs = {
    file.stem: (
        pd.read_excel(file)
        .assign(
            Date=lambda df: pd.to_datetime(
                df["Date"],
                errors="coerce"
            )
        )
        .dropna(subset=["Date"])
        .reset_index(drop=True)
    )
    for file in files
}

total_load_2023.xlsx
total_load_2024.xlsx
total_load_2025.xlsx
total_load_2026.xlsx


In [5]:
for name, df in dfs.items():
    print(
        f"{name}: "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )

display(dfs["total_load_2023"].head())

total_load_2023: 280,320 rows × 4 columns
total_load_2024: 281,088 rows × 4 columns
total_load_2025: 280,320 rows × 4 columns
total_load_2026: 211,936 rows × 4 columns


,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
0,2023-12-31 23:45:00,565.863,560.580,Calabria
1,2023-12-31 23:45:00,1998.771,1980.110,Centre-North
2,2023-12-31 23:45:00,4663.746,4620.205,Centre-South
3,2023-12-31 23:45:00,11502.558,11395.170,North
4,2023-12-31 23:45:00,823.763,816.072,Sardinia


In [6]:
df = pd.concat(
    dfs.values(),
    ignore_index=True
)

df = (
    df.loc[df["Bidding Zone"] == "Sardinia"]
      .reset_index(drop=True)
)

# Assume the first source occurrence of each repeated time is summer time.
dates = df["Date"]
df["Date"] = dates.dt.tz_localize(
    "Europe/Rome",
    ambiguous=(~dates.duplicated(keep="first")).to_numpy(),
)
df = df.sort_values("Date").reset_index(drop=True)

display(df.head())
display(df.tail())

print(f"Rows:  {len(df):,}")
print(f"Start: {df['Date'].min()}")
print(f"End:   {df['Date'].max()}")

,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
0,2023-01-01 00:00:00+01:00,807.675,843.985,Sardinia
1,2023-01-01 00:15:00+01:00,810.231,845.408,Sardinia
2,2023-01-01 00:30:00+01:00,807.468,830.963,Sardinia
3,2023-01-01 00:45:00+01:00,799.173,811.684,Sardinia
4,2023-01-01 01:00:00+01:00,790.115,794.100,Sardinia


,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
131703,2026-10-05 22:45:00+02:00,854.351,854.964,Sardinia
131704,2026-10-05 23:00:00+02:00,830.035,833.345,Sardinia
131705,2026-10-05 23:15:00+02:00,811.984,819.232,Sardinia
131706,2026-10-05 23:30:00+02:00,805.706,809.821,Sardinia
131707,2026-10-05 23:45:00+02:00,782.328,786.290,Sardinia


Rows:  131,708
Start: 2023-01-01 00:00:00+01:00
End:   2026-10-05 23:45:00+02:00


In [7]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate timestamps:")
print(df["Date"].duplicated().sum())

print("\nTime-step distribution:")
print(
    df["Date"]
    .diff()
    .value_counts()
    .sort_index()
)

Missing values:
Date                        0
Total Load [MW]             0
Forecast Total Load [MW]    0
Bidding Zone                0
dtype: int64

Duplicate timestamps:
0

Time-step distribution:
Date
0 days 00:15:00    131706
2 days 00:15:00         1
Name: count, dtype: int64


In [8]:
df[["Total Load [MW]", "Forecast Total Load [MW]"]].describe()

,Total Load [MW],Forecast Total Load [MW]
count,131708.000000,131708.000000
mean,974.587243,980.293774
std,189.494450,189.519653
min,552.930000,567.624000
25%,832.020750,836.768750
50%,972.905500,976.654500
75%,1093.713750,1099.955750
max,1809.779000,1784.726000


In [9]:
diagnostic = df[["Date", "Total Load [MW]", "Forecast Total Load [MW]"]].copy()

diagnostic["dt"] = diagnostic["Date"].diff()
diagnostic["previous_date"] = diagnostic["Date"].shift(1)

anomalies = diagnostic[
    diagnostic["dt"].notna()
    & (diagnostic["dt"] != pd.Timedelta(minutes=15))
]

display(anomalies)

,Date,Total Load [MW],Forecast Total Load [MW],dt,previous_date
106464,2026-01-16 00:00:00+01:00,826.278,861.043,2 days 00:15:00,2026-01-13 23:45:00+01:00


In [10]:
duplicates = df[
    df["Date"].duplicated(keep=False)
].sort_values("Date")

display(duplicates)

,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone


In [11]:
# All physical intervals must be 15 minutes except the January 2026 gap.
assert df["Date"].notna().all()
assert df["Date"].is_unique
assert len(anomalies) == 1, anomalies
assert anomalies["Date"].dt.year.eq(2026).all()
assert anomalies["Date"].dt.month.eq(1).all()
assert anomalies["dt"].gt(pd.Timedelta(days=1)).all()

print("Verified: continuous 15-minute DST transitions; only the January 2026 gap remains.")
display(df["Date"].diff().value_counts().sort_index())


Verified: continuous 15-minute DST transitions; only the January 2026 gap remains.


Date
0 days 00:15:00    131706
2 days 00:15:00         1
Name: count, dtype: int64

In [12]:
hourly = (
    df.set_index("Date")
      [["Total Load [MW]", "Forecast Total Load [MW]"]]
      .resample("1h")
      .mean()
)

print(f"Rows: {len(hourly):,}")
print(f"Start: {hourly.index.min()}")
print(f"End: {hourly.index.max()}")

print("\nMissing values:")
print(hourly.isna().sum())

display(hourly.head())

Rows: 32,975
Start: 2023-01-01 00:00:00+01:00
End: 2026-10-05 23:00:00+02:00

Missing values:
Total Load [MW]             48
Forecast Total Load [MW]    48
dtype: int64


,Total Load [MW],Forecast Total Load [MW]
Date,,
2023-01-01 00:00:00+01:00,806.13675,833.01000
2023-01-01 01:00:00+01:00,783.55525,779.20525
2023-01-01 02:00:00+01:00,753.62675,749.15025
2023-01-01 03:00:00+01:00,736.34950,739.18250
2023-01-01 04:00:00+01:00,723.29950,749.01150


In [13]:
# making sure the gap is in the right time interval

missing = hourly[hourly["Total Load [MW]"].isna()]

print(f"Missing hours: {len(missing)}")
print(f"First missing: {missing.index.min()}")
print(f"Last missing:  {missing.index.max()}")

Missing hours: 48
First missing: 2026-01-14 00:00:00+01:00
Last missing:  2026-01-15 23:00:00+01:00


In [14]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "sardinia_hourly.parquet"

hourly.to_parquet(OUTPUT_PATH)

print(f"Saved to: {OUTPUT_PATH.resolve()}")

Saved to: /Users/aserra/projects/timeseries/energy-load-forecast/data/processed/sardinia_hourly.parquet
